<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/MovieLens_GCN_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# MovieLens-1M internal KG + Relation-aware CCN-Attention (W_r + relation embedding)
# + Multi-negative (1 pos + N neg) InfoNCE / sampled-softmax (cosine)
# + Gradient clipping
# + Recall@{5,10,20,50}
#
# Single-cell runnable in a fresh notebook.

import os, zipfile, urllib.request, random
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Repro & device
# ----------------------------
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ----------------------------
# Download / load MovieLens-1M
# ----------------------------
DATA_DIR = "./data"
ML_NAME  = "ml-1m"
ML_DIR   = os.path.join(DATA_DIR, ML_NAME)
ZIP_PATH = os.path.join(DATA_DIR, f"{ML_NAME}.zip")
URL      = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ML_DIR):
    if not os.path.exists(ZIP_PATH):
        print(f"Downloading: {URL}")
        urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

ratings_path = os.path.join(ML_DIR, "ratings.dat")
movies_path  = os.path.join(ML_DIR, "movies.dat")

ratings_raw = pd.read_csv(
    ratings_path, sep="::", engine="python",
    names=["userId","movieId","rating","timestamp"]
)
ratings_raw.sort_values(["userId","timestamp"], inplace=True)

movies_df = pd.read_csv(
    movies_path, sep="::", engine="python", encoding="latin-1",
    names=["movieId","title","genres"]
)

print("ratings_raw:", ratings_raw.shape, "movies_df:", movies_df.shape)

# Build id maps from FULL ratings
user_ids  = sorted(ratings_raw["userId"].unique().tolist())
movie_ids = sorted(ratings_raw["movieId"].unique().tolist())
uid2u = {uid:i for i,uid in enumerate(user_ids)}
mid2m = {mid:i for i,mid in enumerate(movie_ids)}
num_users  = len(user_ids)
num_movies = len(movie_ids)
print("num_users:", num_users, "num_movies:", num_movies)

# ----------------------------
# Implicit positives + leave-one-out split
# ----------------------------
MIN_POS_RATING = 4.0
pos = ratings_raw[ratings_raw["rating"] >= MIN_POS_RATING].copy()
pos["u"] = pos["userId"].map(uid2u)
pos["m"] = pos["movieId"].map(mid2m)
pos.sort_values(["u","timestamp"], inplace=True)

user_hist = defaultdict(list)
for u, m, ts in pos[["u","m","timestamp"]].itertuples(index=False):
    user_hist[u].append((ts, m))
for u in user_hist:
    user_hist[u].sort(key=lambda x: x[0])

train_pos = defaultdict(list)  # keep list for fast sampling
val_pos   = {}
test_pos  = {}

for u, seq in user_hist.items():
    ms = [m for _,m in seq]
    if len(ms) >= 3:
        test_pos[u] = ms[-1]
        val_pos[u]  = ms[-2]
        train_pos[u] = ms[:-2]
    elif len(ms) == 2:
        test_pos[u] = ms[-1]
        val_pos[u]  = ms[-2]
        train_pos[u] = []  # no train, but can still eval
    elif len(ms) == 1:
        test_pos[u] = ms[-1]
        train_pos[u] = []

train_pos_set = {u: set(train_pos.get(u, [])) for u in range(num_users)}
train_users = [u for u in range(num_users) if len(train_pos.get(u, [])) > 0]
eval_users = [u for u in range(num_users) if u in test_pos]

print("users with test:", len(eval_users), "users with train positives:", len(train_users))

# Train edges for graph construction: user-movie from train only
train_edges = [(u, m) for u in range(num_users) for m in train_pos_set[u]]
print("train_edges:", len(train_edges))
if len(train_edges) == 0:
    raise RuntimeError("No training edges. Try lowering MIN_POS_RATING or changing split.")

# ----------------------------
# Internal KG: movie-genres only
# ----------------------------
movies_sub = movies_df[movies_df["movieId"].isin(movie_ids)].copy()
movies_sub["m"] = movies_sub["movieId"].map(mid2m)

genre_set = set()
movie_genres = defaultdict(list)
for m, gstr in movies_sub[["m","genres"]].itertuples(index=False):
    if isinstance(gstr, str) and gstr.strip():
        for g in gstr.split("|"):
            genre_set.add(g)
            movie_genres[m].append(g)

genres = sorted(list(genre_set))
gid2g = {g:i for i,g in enumerate(genres)}
num_genres = len(genres)
print("num_genres:", num_genres)

# ----------------------------
# Build big graph nodes = [users][movies][genres]
# ----------------------------
OFF_U = 0
OFF_M = OFF_U + num_users
OFF_G = OFF_M + num_movies
num_nodes = OFF_G + num_genres

# Edge types (directed)
REL_U2M = 0
REL_M2U = 1
REL_M2G = 2
REL_G2M = 3
NUM_RELS = 4

edges_src, edges_dst, edges_type = [], [], []
def add_edge(a, b, t):
    edges_src.append(a); edges_dst.append(b); edges_type.append(t)

# user-movie (train) both directions
for u, m in train_edges:
    u_node = OFF_U + u
    m_node = OFF_M + m
    add_edge(u_node, m_node, REL_U2M)
    add_edge(m_node, u_node, REL_M2U)

# movie-genre both directions
for m in range(num_movies):
    m_node = OFF_M + m
    for g in movie_genres.get(m, []):
        g_node = OFF_G + gid2g[g]
        add_edge(m_node, g_node, REL_M2G)
        add_edge(g_node, m_node, REL_G2M)

src = torch.tensor(edges_src, dtype=torch.long, device=device)
dst = torch.tensor(edges_dst, dtype=torch.long, device=device)
etype = torch.tensor(edges_type, dtype=torch.long, device=device)
E = src.numel()
print("num_nodes:", num_nodes, "num_edges(directed):", E)

# ----------------------------
# Stable segment softmax per dst
# ----------------------------
def segment_softmax(e, dst, num_nodes):
    max_per_dst = torch.full((num_nodes,), -float("inf"), device=e.device)
    if hasattr(max_per_dst, "scatter_reduce_"):
        max_per_dst.scatter_reduce_(0, dst, e, reduce="amax", include_self=True)
    else:
        # slow fallback
        for i in range(num_nodes):
            mask = (dst == i)
            if mask.any():
                max_per_dst[i] = torch.max(e[mask])

    e_exp = torch.exp(e - max_per_dst[dst])
    denom = torch.zeros((num_nodes,), device=e.device)
    denom.index_add_(0, dst, e_exp)
    return e_exp / (denom[dst] + 1e-12)

# ----------------------------
# Relation-aware CCN-Attention layer (W_r + r_emb)
# ----------------------------
class RelCCNAttLayer(nn.Module):
    def __init__(self, dim, num_rels, dropout=0.15, negative_slope=0.2, add_self_linear=True):
        super().__init__()
        self.dim = dim
        self.num_rels = num_rels
        self.dropout = dropout
        self.negative_slope = negative_slope

        self.W = nn.ModuleList([nn.Linear(dim, dim, bias=False) for _ in range(num_rels)])
        self.r_emb = nn.Embedding(num_rels, dim)

        # shared attention vectors
        self.a_l = nn.Parameter(torch.empty(dim))
        self.a_r = nn.Parameter(torch.empty(dim))
        self.a_rel = nn.Parameter(torch.empty(dim))

        self.add_self_linear = add_self_linear
        self.self_lin = nn.Linear(dim, dim, bias=True) if add_self_linear else None
        self.reset_parameters()

    def reset_parameters(self):
        for w in self.W:
            nn.init.xavier_uniform_(w.weight)
        nn.init.normal_(self.r_emb.weight, std=0.02)
        nn.init.xavier_uniform_(self.a_l.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_r.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_rel.unsqueeze(0))
        if self.self_lin is not None:
            nn.init.xavier_uniform_(self.self_lin.weight)
            nn.init.zeros_(self.self_lin.bias)

    def forward(self, x, src, dst, etype, num_nodes):
        # precompute per relation transforms
        h_r_all = [w(x) for w in self.W]  # list of [N,D]
        rvec = self.r_emb(etype)          # [E,D]

        h_src = torch.empty((src.size(0), self.dim), device=x.device, dtype=x.dtype)
        h_dst = torch.empty((src.size(0), self.dim), device=x.device, dtype=x.dtype)

        # small NUM_RELS -> mask fill
        for r in range(self.num_rels):
            mask = (etype == r)
            if mask.any():
                hr = h_r_all[r]
                h_src[mask] = hr[src[mask]]
                h_dst[mask] = hr[dst[mask]]

        # attention logits
        e = F.leaky_relu(
            (h_src * self.a_l).sum(-1) + (h_dst * self.a_r).sum(-1) + (rvec * self.a_rel).sum(-1),
            negative_slope=self.negative_slope
        )

        alpha = segment_softmax(e, dst, num_nodes)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # message
        msg = h_src + rvec
        out = torch.zeros((num_nodes, self.dim), device=x.device, dtype=x.dtype)
        out.index_add_(0, dst, msg * alpha.unsqueeze(-1))

        if self.self_lin is not None:
            out = out + self.self_lin(x)
        return out

class KGRecRelAtt(nn.Module):
    def __init__(self, num_nodes, num_users, num_movies, off_u, off_m, dim=96, layers=2, dropout=0.15):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, dim)
        nn.init.normal_(self.emb.weight, std=0.01)
        self.layers = nn.ModuleList([
            RelCCNAttLayer(dim=dim, num_rels=NUM_RELS, dropout=dropout, add_self_linear=True)
            for _ in range(layers)
        ])
        self.num_users = num_users
        self.num_movies = num_movies
        self.off_u = off_u
        self.off_m = off_m

    def forward(self, src, dst, etype, num_nodes):
        x = self.emb.weight
        xs = [x]
        for layer in self.layers:
            x = layer(x, src, dst, etype, num_nodes)
            x = F.relu(x)
            xs.append(x)
        x_final = torch.mean(torch.stack(xs, dim=0), dim=0)

        user_e  = x_final[self.off_u:self.off_u + self.num_users]
        movie_e = x_final[self.off_m:self.off_m + self.num_movies]
        return user_e, movie_e

model = KGRecRelAtt(
    num_nodes=num_nodes,
    num_users=num_users,
    num_movies=num_movies,
    off_u=OFF_U,
    off_m=OFF_M,
    dim=96,
    layers=2,
    dropout=0.15
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)

# ----------------------------
# Multi-negative sampling & InfoNCE loss (cosine)
# ----------------------------
NEG_PER_POS = 100
TEMPERATURE = 0.07  # common for cosine contrastive
EPS = 1e-12

def sample_negs_for_user(u, n):
    blocked = set(train_pos_set[u])
    if u in val_pos:  blocked.add(val_pos[u])
    if u in test_pos: blocked.add(test_pos[u])
    negs = []
    while len(negs) < n:
        m = int(np.random.randint(0, num_movies))
        if m not in blocked:
            negs.append(m)
    return np.array(negs, dtype=np.int64)

def info_nce_loss(u_e, pos_e, neg_e, temperature=TEMPERATURE, eps=EPS):
    """
    u_e:   [B,D]
    pos_e: [B,D]
    neg_e: [B,N,D]
    """
    u   = F.normalize(u_e,   p=2, dim=-1, eps=eps)
    pos = F.normalize(pos_e, p=2, dim=-1, eps=eps)
    neg = F.normalize(neg_e, p=2, dim=-1, eps=eps)

    pos_s = (u * pos).sum(dim=-1, keepdim=True)              # [B,1]
    neg_s = (u.unsqueeze(1) * neg).sum(dim=-1)               # [B,N]
    logits = torch.cat([pos_s, neg_s], dim=1) / temperature  # [B,1+N]
    labels = torch.zeros((logits.size(0),), device=logits.device, dtype=torch.long)
    return F.cross_entropy(logits, labels)

# ----------------------------
# Recall@K (cosine)
# ----------------------------
@torch.no_grad()
def recall_at_ks(user_e, movie_e, ks=(5,10,20,50), users=None, eps=EPS):
    if users is None:
        users = eval_users

    user_e  = F.normalize(user_e,  p=2, dim=-1, eps=eps)
    movie_e = F.normalize(movie_e, p=2, dim=-1, eps=eps)
    movie_e_t = movie_e.t()

    recalls = {k: [] for k in ks}
    for u in users:
        gt = test_pos.get(u, None)
        if gt is None:
            continue

        scores = (user_e[u:u+1] @ movie_e_t).squeeze(0)

        seen = set(train_pos_set[u])
        if u in val_pos: seen.add(val_pos[u])
        if len(seen) > 0:
            scores[torch.tensor(list(seen), device=device)] = -1e9

        topk = torch.topk(scores, k=min(max(ks), num_movies)).indices.detach().cpu().numpy()
        for k in ks:
            recalls[k].append(1.0 if gt in topk[:k] else 0.0)

    return {k: float(np.mean(recalls[k])) if len(recalls[k]) else 0.0 for k in ks}

# ----------------------------
# Training (ONE backward per epoch) + grad clip
# ----------------------------
BATCH = 1024
EPOCHS = 100
EVAL_EVERY = 1
CLIP_NORM = 0.5

# how many steps per epoch
steps = max(1, len(train_edges) // BATCH)

print("\nTraining...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    user_e, movie_e = model(src, dst, etype, num_nodes)  # one autograd graph for epoch

    total_loss = 0.0
    for _ in range(steps):
        us = np.random.choice(train_users, size=BATCH, replace=True)

        pos_ms = []
        neg_mx = []
        for u in us:
            pos_list = train_pos[u]
            pm = int(pos_list[np.random.randint(0, len(pos_list))])
            negs = sample_negs_for_user(u, NEG_PER_POS)
            pos_ms.append(pm)
            neg_mx.append(negs)

        us_t  = torch.tensor(us, device=device, dtype=torch.long)
        pos_t = torch.tensor(pos_ms, device=device, dtype=torch.long)
        neg_t = torch.tensor(np.stack(neg_mx, axis=0), device=device, dtype=torch.long)  # [B,N]

        uvec = user_e[us_t]                # [B,D]
        pvec = movie_e[pos_t]              # [B,D]
        nvec = movie_e[neg_t]              # [B,N,D]

        total_loss = total_loss + info_nce_loss(uvec, pvec, nvec)

    total_loss = total_loss / steps
    opt.zero_grad(set_to_none=True)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_NORM)
    opt.step()

    if epoch % EVAL_EVERY == 0:
        model.eval()
        ue, me = model(src, dst, etype, num_nodes)
        rec = recall_at_ks(ue, me, ks=(5,10,20,50), users=eval_users)
        print(f"Epoch {epoch:02d} | loss={float(total_loss.detach().cpu()):.4f} | "
              f"R@5={rec[5]:.4f} R@10={rec[10]:.4f} R@20={rec[20]:.4f} R@50={rec[50]:.4f}")

print("\nDone.")

device: cuda
ratings_raw: (1000209, 4) movies_df: (3883, 3)
num_users: 6040 num_movies: 3706
users with test: 6038 users with train positives: 6035
train_edges: 563206
num_genres: 18
num_nodes: 9764 num_edges(directed): 1138796

Training...
Epoch 01 | loss=4.7492 | R@5=0.0048 R@10=0.0106 R@20=0.0234 R@50=0.0568
Epoch 02 | loss=4.1447 | R@5=0.0089 R@10=0.0162 R@20=0.0301 R@50=0.0775
Epoch 03 | loss=3.8024 | R@5=0.0109 R@10=0.0210 R@20=0.0386 R@50=0.0919
Epoch 04 | loss=3.5483 | R@5=0.0121 R@10=0.0260 R@20=0.0485 R@50=0.1100
Epoch 05 | loss=3.4114 | R@5=0.0167 R@10=0.0301 R@20=0.0568 R@50=0.1285
Epoch 06 | loss=3.3500 | R@5=0.0169 R@10=0.0336 R@20=0.0672 R@50=0.1414
Epoch 07 | loss=3.3000 | R@5=0.0187 R@10=0.0381 R@20=0.0704 R@50=0.1507
Epoch 08 | loss=3.2496 | R@5=0.0190 R@10=0.0391 R@20=0.0749 R@50=0.1530
Epoch 09 | loss=3.2149 | R@5=0.0205 R@10=0.0396 R@20=0.0754 R@50=0.1522
Epoch 10 | loss=3.1808 | R@5=0.0222 R@10=0.0389 R@20=0.0778 R@50=0.1507
Epoch 11 | loss=3.1415 | R@5=0.0227 R@1